In [ ]:
# =============================================================================
# CELL 1 -- imports and configuration
# =============================================================================
import os, json, time, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import joblib
from itertools import product
from scipy.optimize import minimize

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.feature_selection import SelectFromModel
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             matthews_corrcoef, precision_score, recall_score,
                             confusion_matrix, roc_auc_score,
                             average_precision_score, log_loss)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# ---------------- paths ----------------
TRAIN_PATH = "/kaggle/input/datasets/hsharmaa/foodev-v3-10299/features_prott5_10639.csv"
TEST_PATH  = "/kaggle/input/datasets/hsharmaa/foodev-v3-10299/features_prott5_2660.csv"
OUTPUT_DIR = "/kaggle/working/foodevpred_final"
MODEL_DIR  = os.path.join(OUTPUT_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

ID_COL, TARGET_COL, STRAT_COL = "seq_id", "primary_label", "secondary_label"
CLASS_NAMES = {0: "Non_EV", 1: "Milk_EV", 2: "Plant_EV"}
N_CLS = 3

# ---------------- FINAL CONFIGURATION (from final_model_card.json) ------------
SELECT_K   = 384
BASE_ORDER = ["LightGBM", "SVM_RBF", "MLP", "KNN_cosine"]
BASE_CFG   = {"LightGBM": "reg", "SVM_RBF": "C2", "MLP": "reg", "KNN_cosine": "k15"}
META_SETTING = "ET_d10_leaf40"
RESTACK      = False
CALIBRATE    = True
ENRICH_META  = True
DECISION_OBJECTIVE = "acc"
SEED, N_INNER, N_OUTER, N_SEEDS = 42, 3, 5, 3
INNER_VAL_FRAC = 0.20
DETERMINISTIC  = {"KNN_cosine"}

# ---------------- audit switches ----------------
RUN_CV_AUDIT  = True    # nested 5x3 CV of the final pipeline (the overfitting proof)
LOW_VARIANCE  = False   # True -> XGB_meta (gap 8.35) instead of ET_d10_leaf40
ALSO_REPORT_BACC_POINT = True   # same model, decision weights tuned for balanced acc

print("FoodEVPred v3 — final standalone")
print(f"  bases   : {BASE_ORDER}")
print(f"  variants: {BASE_CFG}")
print(f"  meta    : {'XGB_meta (low-variance)' if LOW_VARIANCE else META_SETTING}")
print(f"  models -> {MODEL_DIR}")

In [ ]:
# =============================================================================
# CELL 2 -- metrics (identical definitions to the staged notebooks)
# =============================================================================
def get_metrics(y_true, y_pred, y_score=None, n_classes=N_CLS):
    m = {"acc": accuracy_score(y_true, y_pred),
         "bacc": balanced_accuracy_score(y_true, y_pred),
         "f1": f1_score(y_true, y_pred, average="macro"),
         "pre": precision_score(y_true, y_pred, average="macro", zero_division=0),
         "sens": recall_score(y_true, y_pred, average="macro", zero_division=0),
         "mcc": matthews_corrcoef(y_true, y_pred)}
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    specs, npvs = [], []
    for i in range(n_classes):
        tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)
        specs.append(tn / (tn + fp) if (tn + fp) else 0.0)
        npvs.append(tn / (tn + fn) if (tn + fn) else 0.0)
        m[f"sens_c{i}"] = tp / (tp + fn) if (tp + fn) else 0.0
        m[f"spec_c{i}"] = specs[-1]
        m[f"pre_c{i}"]  = tp / (tp + fp) if (tp + fp) else 0.0
        m[f"npv_c{i}"]  = npvs[-1]
    m["spec"] = float(np.mean(specs)); m["npv"] = float(np.mean(npvs))
    if y_score is not None:
        yb = label_binarize(y_true, classes=list(range(n_classes)))
        try:    m["auc"] = roc_auc_score(y_true, y_score, multi_class="ovr",
                                         average="macro", labels=list(range(n_classes)))
        except Exception: m["auc"] = np.nan
        try:    m["ap"] = average_precision_score(yb, y_score, average="macro")
        except Exception: m["ap"] = np.nan
    else:
        m["auc"] = m["ap"] = np.nan
    return m

def metric_row(y_true, P, pred, label):
    m = get_metrics(y_true, pred, P)
    row = {"split": label, "n": len(y_true)}
    for k, lbl in [("acc","Accuracy"),("bacc","Balanced accuracy"),
                   ("sens","Sensitivity (macro)"),("spec","Specificity (macro)"),
                   ("pre","Precision (macro)"),("npv","NPV (macro)"),("f1","macro F1")]:
        row[lbl] = m[k] * 100
    row["MCC"] = m["mcc"]; row["AUC (macro OvR)"] = m["auc"]; row["AP (macro OvR)"] = m["ap"]
    row["log-loss"] = log_loss(y_true, np.clip(P, 1e-9, 1), labels=list(range(N_CLS)))
    for i in range(N_CLS):
        row[f"Sens {CLASS_NAMES[i]}"] = m[f"sens_c{i}"] * 100
        row[f"PPV {CLASS_NAMES[i]}"]  = m[f"pre_c{i}"] * 100
    return row

In [ ]:
# =============================================================================
# CELL 3 -- pipeline components
# =============================================================================
# ClassifierMixin MUST precede BaseEstimator: sklearn >=1.6 derives is_classifier
# from the tag system and CalibratedClassifierCV rejects the reverse order.
class BalancedWrapper(ClassifierMixin, BaseEstimator):
    def __init__(self, estimator=None):
        self.estimator = estimator
    def fit(self, X, y, sample_weight=None):
        w = compute_sample_weight("balanced", y) if sample_weight is None else sample_weight
        self.estimator_ = clone(self.estimator).fit(X, y, sample_weight=w)
        self.classes_ = self.estimator_.classes_
        return self
    def predict_proba(self, X): return self.estimator_.predict_proba(X)
    def predict(self, X):       return self.estimator_.predict(X)


def make_selector(seed=SEED):
    """LGBM_gain: split-gain importance over the full 1024-dim ProtT5 block."""
    return SelectFromModel(
        LGBMClassifier(n_estimators=300, max_depth=6, importance_type="gain",
                       class_weight="balanced", verbosity=-1,
                       random_state=seed, n_jobs=-1),
        max_features=SELECT_K, threshold=-np.inf)


def make_model(name, variant, seed):
    if name == "LightGBM":
        p = dict(n_estimators=400, learning_rate=0.05, max_depth=6, num_leaves=15,
                 min_child_samples=40, subsample=0.8, subsample_freq=1,
                 colsample_bytree=0.5, reg_lambda=5.0, class_weight="balanced",
                 verbosity=-1, random_state=seed, n_jobs=-1)
        if variant == "reg":
            p.update(num_leaves=7, min_child_samples=80, reg_lambda=10.0,
                     colsample_bytree=0.4)
        elif variant == "deep":
            p.update(num_leaves=31, max_depth=10, n_estimators=600, learning_rate=0.03)
        return LGBMClassifier(**p)
    if name == "SVM_RBF":
        C = {"C2": 2.0, "C5": 5.0, "C20": 20.0}[variant]
        # probability=False under calibration: True would nest libsvm's own 5-fold
        # Platt CV inside the calibrator's 3-fold CV (15x cost, worse calibration).
        return SVC(C=C, kernel="rbf", gamma="scale", probability=not CALIBRATE,
                   class_weight="balanced", random_state=seed)
    if name == "MLP":
        p = dict(hidden_layer_sizes=(512, 256, 128), alpha=1e-3, max_iter=500,
                 early_stopping=True, n_iter_no_change=15,
                 validation_fraction=INNER_VAL_FRAC, random_state=seed)
        if variant == "reg":
            p.update(hidden_layer_sizes=(256, 128), alpha=1e-2)
        return MLPClassifier(**p)
    if name == "KNN_cosine":
        k = {"k15": 15, "k25": 25, "k45": 45}[variant]
        return KNeighborsClassifier(n_neighbors=k, weights="distance",
                                    metric="cosine", n_jobs=-1)
    raise ValueError(name)


def make_meta():
    if LOW_VARIANCE:
        # CV 81.44 with an 8.35-point train gap instead of 11.25
        return XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                             reg_lambda=5.0, subsample=0.8, colsample_bytree=0.8,
                             eval_metric="mlogloss", tree_method="hist",
                             random_state=SEED, n_jobs=-1)
    return ExtraTreesClassifier(n_estimators=400, max_depth=10, min_samples_leaf=40,
                                class_weight="balanced", random_state=SEED, n_jobs=-1)


def fit_bases(A, yv, n_seeds=N_SEEDS):
    """Fit every base under n_seeds; return {name: [fitted estimators]}."""
    out = {}
    for b in BASE_ORDER:
        seeds = [SEED] if b in DETERMINISTIC else [SEED + 1000 * s for s in range(n_seeds)]
        ests = []
        for sd in seeds:
            e = make_model(b, BASE_CFG[b], sd)
            if CALIBRATE:
                e = CalibratedClassifierCV(e, method="isotonic", cv=3)
            ests.append(e.fit(A, yv))
        out[b] = ests
    return out


def base_proba(ests, Z, n_cls=N_CLS):
    """Seed-averaged, class-complete probabilities."""
    out = np.zeros((len(Z), n_cls))
    for e in ests:
        P = e.predict_proba(Z)
        for j, c in enumerate(e.classes_):
            out[:, int(c)] += P[:, j] / len(ests)
    return out


def enrich(blocks, n_cls=N_CLS):
    """Meta matrix: probabilities + confidence + entropy + cross-model agreement.
    A strict function of the base probabilities -- no new feature source."""
    parts = [np.hstack(blocks)]
    if ENRICH_META:
        for B in blocks:
            Bc = np.clip(B, 1e-9, 1.0)
            parts.append(Bc.max(axis=1, keepdims=True))
            parts.append(-(Bc * np.log(Bc)).sum(axis=1, keepdims=True))
        votes = np.stack([np.argmax(B, axis=1) for B in blocks], axis=1)
        parts.append(np.stack([(votes == c).mean(axis=1) for c in range(n_cls)], axis=1))
    return np.hstack(parts)


# ---- decision layer ----------------------------------------------------------
# Bases are fitted with class_weight="balanced", which re-prices the classes to a
# uniform prior. That maximises BALANCED accuracy and costs PLAIN accuracy on a
# 72%-majority set. The fix is not to drop the weighting (it protects minority
# recall) but to put the prior back once, explicitly, at the decision step:
# three non-negative multipliers fitted on OUT-OF-FOLD training predictions only.
# w = (1,1,1) recovers plain argmax, so the layer can only help or be neutral.
def _obj(y_true, P, w, objective):
    pred = np.argmax(P * w[None, :], axis=1)
    if objective == "acc":  return -accuracy_score(y_true, pred)
    if objective == "bacc": return -balanced_accuracy_score(y_true, pred)
    return -f1_score(y_true, pred, average="macro")

def fit_decision_weights(y_oof, P_oof, objective="acc", n_classes=N_CLS, seed=SEED):
    best_w, best_v = np.ones(n_classes), _obj(y_oof, P_oof, np.ones(n_classes), objective)
    rs = np.random.RandomState(seed)
    for s in [np.zeros(n_classes)] + [rs.normal(0, 0.35, n_classes) for _ in range(6)]:
        r = minimize(lambda t: _obj(y_oof, P_oof, np.exp(t), objective), s,
                     method="Nelder-Mead",
                     options=dict(maxiter=600, xatol=1e-3, fatol=1e-5))
        if r.fun < best_v:
            best_v, best_w = r.fun, np.exp(r.x)
    return best_w / best_w.sum() * n_classes

def apply_decision(P, w):
    return np.argmax(P * w[None, :], axis=1)

In [ ]:
# =============================================================================
# CELL 4 -- load data
# =============================================================================
tr = pd.read_csv(TRAIN_PATH)
te = pd.read_csv(TEST_PATH)
dim_cols = [c for c in tr.columns if c.startswith("dim_")]
if len(dim_cols) < 500:
    raise ValueError("Point TRAIN_PATH at the FULL 1024-dim ProtT5 file: the "
                     "LGBM_gain selection is refitted inside the folds here.")

X      = tr[dim_cols].to_numpy(np.float32)
y      = tr[TARGET_COL].to_numpy()
strat  = tr[STRAT_COL].to_numpy()
X_test = te[dim_cols].to_numpy(np.float32)
y_test = te[TARGET_COL].to_numpy()

print(f"train {X.shape}  classes {np.bincount(y)}  majority "
      f"{np.bincount(y).max()/len(y)*100:.2f}%")
print(f"test  {X_test.shape}  classes {np.bincount(y_test)}")

In [ ]:
# =============================================================================
# CELL 5 -- OVERFITTING AUDIT: nested 5 x 3 CV of the complete pipeline
# =============================================================================
# Every supervised step -- the LGBM_gain selection, the isotonic calibrators, the
# bases, the meta-learner and the decision weights -- is refitted inside each
# outer fold. Nothing from the held-out fold, and nothing from the locked test
# set, informs any of them. This is the number the external test is compared
# against; if the test result lands inside this interval, selection did not
# overfit, whatever the train-vs-CV gap looks like.
def run_fold(Xtr, ytr, strtr, Xte_, n_seeds=N_SEEDS):
    sel = make_selector().fit(Xtr, ytr)
    A, B = sel.transform(Xtr), sel.transform(Xte_)
    sc = StandardScaler().fit(A); A, B = sc.transform(A), sc.transform(B)

    oof = {b: np.zeros((len(ytr), N_CLS)) for b in BASE_ORDER}
    for i_tr, i_va in StratifiedKFold(N_INNER, shuffle=True,
                                      random_state=SEED).split(A, strtr):
        f = fit_bases(A[i_tr], ytr[i_tr], n_seeds)
        for b in BASE_ORDER:
            oof[b][i_va] = base_proba(f[b], A[i_va])

    fitted = fit_bases(A, ytr, n_seeds)
    ins = {b: base_proba(fitted[b], A) for b in BASE_ORDER}
    tst = {b: base_proba(fitted[b], B) for b in BASE_ORDER}

    meta = make_meta()
    meta.fit(enrich([oof[b] for b in BASE_ORDER]), ytr)
    def pp(M):
        q = meta.predict_proba(M); P = np.zeros((M.shape[0], N_CLS))
        for j, c in enumerate(meta.classes_):
            P[:, int(c)] = q[:, j]
        return P
    P_oof = pp(enrich([oof[b] for b in BASE_ORDER]))
    P_ins = pp(enrich([ins[b] for b in BASE_ORDER]))
    P_tst = pp(enrich([tst[b] for b in BASE_ORDER]))
    w = fit_decision_weights(ytr, P_oof, DECISION_OBJECTIVE)
    return P_ins, P_tst, w

audit = None
if RUN_CV_AUDIT:
    t0, rows = time.time(), []
    for k, (i_tr, i_te) in enumerate(StratifiedKFold(
            N_OUTER, shuffle=True, random_state=SEED).split(X, strat)):
        P_ins, P_tst, w = run_fold(X[i_tr], y[i_tr], strat[i_tr], X[i_te])
        rows.append({
            "fold": k + 1,
            "train_ACC": accuracy_score(y[i_tr], apply_decision(P_ins, w)) * 100,
            "cv_ACC":    accuracy_score(y[i_te], apply_decision(P_tst, w)) * 100,
            "cv_BACC":   balanced_accuracy_score(y[i_te], apply_decision(P_tst, w)) * 100,
            "cv_F1":     f1_score(y[i_te], apply_decision(P_tst, w), average="macro") * 100,
            "cv_AUC":    roc_auc_score(y[i_te], P_tst, multi_class="ovr",
                                       average="macro", labels=list(range(N_CLS)))})
        print(f"  fold {k+1}/{N_OUTER}  train {rows[-1]['train_ACC']:.2f}  "
              f"cv {rows[-1]['cv_ACC']:.2f}  [{time.time()-t0:.0f}s]")
    audit = pd.DataFrame(rows)
    mu, sd = audit.cv_ACC.mean(), audit.cv_ACC.std(ddof=1)
    se = sd / np.sqrt(len(audit))
    CV_LO, CV_HI = mu - 1.96 * se, mu + 1.96 * se
    print(f"\n  nested CV accuracy  {mu:.2f} +/- {sd:.2f}   95% CI [{CV_LO:.2f}, {CV_HI:.2f}]")
    print(f"  mean training accuracy {audit.train_ACC.mean():.2f}"
          f"   train-CV gap {audit.train_ACC.mean()-mu:.2f}")
    audit.to_csv(os.path.join(OUTPUT_DIR, "cv_audit_folds.csv"), index=False)
else:
    CV_LO, CV_HI, mu = 80.92, 82.22, 81.57      # values from the staged run
    print("CV audit skipped -- using the staged-run interval [80.92, 82.22]")

In [ ]:
# =============================================================================
# CELL 6 -- fit the FINAL production model on the whole training split
# =============================================================================
t0 = time.time()
SELECTOR = make_selector().fit(X, y)
FEAT_NAMES = np.asarray(dim_cols)[SELECTOR.get_support()]
A, B = SELECTOR.transform(X), SELECTOR.transform(X_test)
SCALER = StandardScaler().fit(A)
A, B = SCALER.transform(A), SCALER.transform(B)
print(f"  selection + scaling: {A.shape}  [{time.time()-t0:.0f}s]")

# honest out-of-fold matrix -> the meta-learner never sees a base prediction that
# was made on a row that base was fitted on
OOF = {b: np.zeros((len(y), N_CLS)) for b in BASE_ORDER}
for j, (i_tr, i_va) in enumerate(StratifiedKFold(
        N_INNER, shuffle=True, random_state=SEED).split(A, strat)):
    f = fit_bases(A[i_tr], y[i_tr])
    for b in BASE_ORDER:
        OOF[b][i_va] = base_proba(f[b], A[i_va])
    print(f"  OOF inner {j+1}/{N_INNER}  [{time.time()-t0:.0f}s]")

FITTED = fit_bases(A, y)
INS = {b: base_proba(FITTED[b], A) for b in BASE_ORDER}
TST = {b: base_proba(FITTED[b], B) for b in BASE_ORDER}
print(f"  production bases fitted  [{time.time()-t0:.0f}s]")

META = make_meta()
META.fit(enrich([OOF[b] for b in BASE_ORDER]), y)
def stack_pp(M):
    q = META.predict_proba(M); P = np.zeros((M.shape[0], N_CLS))
    for j, c in enumerate(META.classes_):
        P[:, int(c)] = q[:, j]
    return P

P_OOF  = stack_pp(enrich([OOF[b] for b in BASE_ORDER]))
P_INS  = stack_pp(enrich([INS[b] for b in BASE_ORDER]))
P_TEST = stack_pp(enrich([TST[b] for b in BASE_ORDER]))

W_ACC  = fit_decision_weights(y, P_OOF, "acc")
W_BACC = fit_decision_weights(y, P_OOF, "bacc")
W_DEC  = W_ACC if DECISION_OBJECTIVE == "acc" else W_BACC
print(f"\n  decision weights (accuracy point)          : {np.round(W_ACC,3)}")
print(f"  decision weights (balanced-accuracy point) : {np.round(W_BACC,3)}")
print(f"  total fit time {time.time()-t0:.0f}s")

In [ ]:
# =============================================================================
# CELL 7 -- FINAL EVALUATION: training vs external test
# =============================================================================
# Three training-side numbers, and the distinction matters:
#   in-sample   -- bases and meta scoring rows they were fitted on. Always
#                  optimistic; reported only so the size of that optimism shows.
#   out-of-fold -- the HONEST training-side number, from the matrix the meta was
#                  fitted on. THIS is the fair comparator for the test set.
#   nested CV   -- the selection-clean estimate from cell 5.
rows = [metric_row(y, P_INS,  apply_decision(P_INS,  W_DEC), "Training (in-sample)"),
        metric_row(y, P_OOF,  apply_decision(P_OOF,  W_DEC), "Training (out-of-fold)"),
        metric_row(y_test, P_TEST, apply_decision(P_TEST, W_DEC), "External test (locked)")]
EV = pd.DataFrame(rows).set_index("split")
EV.loc["Delta (test - train OOF)"] = (EV.loc["External test (locked)"]
                                      - EV.loc["Training (out-of-fold)"])
EV.loc["Delta (test - train OOF)", "n"] = np.nan
EV.round(3).to_csv(os.path.join(OUTPUT_DIR, "final_evaluation_train_vs_test.csv"))
print(EV.round(2).to_string())

test_acc = EV.loc["External test (locked)", "Accuracy"]
print(f"\n{'='*72}\n  OVERFITTING VERDICT\n{'='*72}")
print(f"  training accuracy (in-sample) : {EV.loc['Training (in-sample)','Accuracy']:.2f}")
print(f"  training accuracy (OOF)       : {EV.loc['Training (out-of-fold)','Accuracy']:.2f}")
print(f"  nested CV accuracy            : {mu:.2f}   95% CI [{CV_LO:.2f}, {CV_HI:.2f}]")
print(f"  external test accuracy        : {test_acc:.2f}")
inside = CV_LO <= test_acc <= CV_HI
print(f"\n  external test inside the nested-CV interval : {inside}")
print("""
  A model that had overfit its own selection would score BELOW its
  cross-validated estimate on untouched data. The in-sample/CV gap measures
  base-learner capacity -- LightGBM and a 3-layer MLP memorise their training
  rows -- not leakage, because every number used for selection came from
  out-of-fold predictions. No corrective action is required.""" if inside else """
  The test result falls OUTSIDE the nested-CV interval. Investigate before
  reporting: check the split provenance and whether any supervised step was
  fitted outside its fold.""")

if ALSO_REPORT_BACC_POINT:
    print(f"\n{'='*72}\n  THE SAME MODEL AT TWO OPERATING POINTS\n{'='*72}")
    op = []
    for nm, w in [("accuracy-optimised", W_ACC), ("balanced-accuracy-optimised", W_BACC)]:
        r = metric_row(y_test, P_TEST, apply_decision(P_TEST, w), nm)
        r["weights"] = np.round(w, 3).tolist()
        op.append(r)
    OP = pd.DataFrame(op).set_index("split")
    keep = ["Accuracy", "Balanced accuracy", "macro F1", "MCC"] + \
           [f"Sens {CLASS_NAMES[i]}" for i in range(N_CLS)] + ["weights"]
    print(OP[keep].round(2).to_string())
    OP.round(3).to_csv(os.path.join(OUTPUT_DIR, "operating_points.csv"))
    print("""
  Only the three decision weights differ -- the trained model is identical.
  The accuracy point buys its headline number largely on the majority class;
  read the per-class sensitivities before choosing which to lead with, and
  report both in the manuscript.""")

In [ ]:
# =============================================================================
# CELL 8 -- save every component as .pkl
# =============================================================================
# Saved separately AND as one bundle. Separate files allow inspecting any
# single learner; the bundle is what the web server should load.
saved = []
def dump(obj, name):
    p = os.path.join(MODEL_DIR, name)
    joblib.dump(obj, p, compress=3)
    saved.append((name, os.path.getsize(p) / 1e6))
    return p

dump(SELECTOR, "01_feature_selector_lgbm_gain_k384.pkl")
dump(SCALER,   "02_standard_scaler.pkl")
for b in BASE_ORDER:
    dump(FITTED[b], f"03_base_{b}.pkl")          # list of seed-bagged estimators
dump(META, "04_meta_extratrees.pkl" if not LOW_VARIANCE else "04_meta_xgb.pkl")

BUNDLE = {
    "version": "FoodEVPred-v3-final",
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "selector": SELECTOR, "scaler": SCALER,
    "bases": FITTED, "base_order": BASE_ORDER, "base_config": BASE_CFG,
    "meta": META, "meta_setting": "XGB_meta" if LOW_VARIANCE else META_SETTING,
    "decision_weights_acc": W_ACC, "decision_weights_bacc": W_BACC,
    "decision_weights": W_DEC, "decision_objective": DECISION_OBJECTIVE,
    "enrich_meta": ENRICH_META, "restack": RESTACK, "calibrate": CALIBRATE,
    "select_k": SELECT_K, "feature_names": list(FEAT_NAMES),
    "dim_cols": dim_cols, "class_names": CLASS_NAMES,
    "n_seeds": N_SEEDS, "seed": SEED, "n_inner": N_INNER,
}
dump(BUNDLE, "foodevpred_v3_final_bundle.pkl")

print("Saved:")
for n, mb in saved:
    print(f"   {n:<46} {mb:7.2f} MB")

# ---- inference helper, written to disk so the web server has no notebook dep --
HELPER = '''"""FoodEVPred v3 - inference helper.

    import joblib, numpy as np, pandas as pd
    from foodevpred_predict import predict_proba, predict

    bundle = joblib.load("foodevpred_v3_final_bundle.pkl")
    df = pd.read_csv("new_prott5_features.csv")      # must contain dim_0..dim_1023
    P    = predict_proba(bundle, df)                  # (n, 3)
    yhat = predict(bundle, df)                        # 0=Non_EV 1=Milk_EV 2=Plant_EV
"""
import numpy as np


def _base_proba(ests, Z, n_cls=3):
    out = np.zeros((len(Z), n_cls))
    for e in ests:
        P = e.predict_proba(Z)
        for j, c in enumerate(e.classes_):
            out[:, int(c)] += P[:, j] / len(ests)
    return out


def _enrich(blocks, enrich_meta=True, n_cls=3):
    parts = [np.hstack(blocks)]
    if enrich_meta:
        for B in blocks:
            Bc = np.clip(B, 1e-9, 1.0)
            parts.append(Bc.max(axis=1, keepdims=True))
            parts.append(-(Bc * np.log(Bc)).sum(axis=1, keepdims=True))
        votes = np.stack([np.argmax(B, axis=1) for B in blocks], axis=1)
        parts.append(np.stack([(votes == c).mean(axis=1) for c in range(n_cls)], axis=1))
    return np.hstack(parts)


def predict_proba(bundle, X):
    """X: DataFrame with the 1024 dim_ columns, or an (n, 1024) array."""
    if hasattr(X, "columns"):
        X = X[bundle["dim_cols"]].to_numpy(np.float32)
    X = np.asarray(X, dtype=np.float32)
    Z = bundle["scaler"].transform(bundle["selector"].transform(X))
    blocks = [_base_proba(bundle["bases"][b], Z) for b in bundle["base_order"]]
    M = _enrich(blocks, bundle["enrich_meta"])
    q = bundle["meta"].predict_proba(M)
    P = np.zeros((len(Z), 3))
    for j, c in enumerate(bundle["meta"].classes_):
        P[:, int(c)] = q[:, j]
    return P


def predict(bundle, X, operating_point="acc"):
    w = bundle["decision_weights_bacc"] if operating_point == "bacc" \\
        else bundle["decision_weights_acc"]
    return np.argmax(predict_proba(bundle, X) * w[None, :], axis=1)
'''
hp = os.path.join(OUTPUT_DIR, "foodevpred_predict.py")
with open(hp, "w") as fh:
    fh.write(HELPER)
print(f"\n   foodevpred_predict.py written to {OUTPUT_DIR}")

# ---- round-trip check: reload from disk and reproduce P_TEST -----------------
import importlib.util
spec = importlib.util.spec_from_file_location("foodevpred_predict", hp)
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
bundle_rt = joblib.load(os.path.join(MODEL_DIR, "foodevpred_v3_final_bundle.pkl"))
P_rt = mod.predict_proba(bundle_rt, te)
delta = float(np.abs(P_rt - P_TEST).max())
print(f"\n   round-trip max |reloaded - in-memory| = {delta:.3e}")
if delta > 1e-8:
    raise RuntimeError("Saved bundle does not reproduce the in-memory model.")
print("   PASS -- the saved bundle reproduces the model exactly.")

json.dump({"selection_method": "LGBM_gain", "select_k": SELECT_K,
           "base_order": BASE_ORDER, "base_config": BASE_CFG,
           "meta_setting": "XGB_meta" if LOW_VARIANCE else META_SETTING,
           "restack": RESTACK, "calibrate": CALIBRATE, "enrich_meta": ENRICH_META,
           "decision_weights_acc": W_ACC.tolist(),
           "decision_weights_bacc": W_BACC.tolist(),
           "decision_objective": DECISION_OBJECTIVE,
           "n_seeds": N_SEEDS, "seed": SEED, "n_inner": N_INNER, "n_outer": N_OUTER,
           "nested_cv_accuracy": float(mu),
           "nested_cv_ci95": [float(CV_LO), float(CV_HI)],
           "external_test_accuracy": float(test_acc)},
          open(os.path.join(OUTPUT_DIR, "final_model_card.json"), "w"), indent=2)
print(f"\nAll artefacts in {OUTPUT_DIR}")

In [ ]:
%cd /kaggle/working
!zip -r foodevpred_final_st.zip foodevpred_final